In [153]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("manarm7md/maqa-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'maqa-dataset' dataset.
Path to dataset files: /kaggle/input/maqa-dataset


In [154]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    SimpleRNN,
    Dense,
    Dropout
)

from tensorflow.keras.callbacks import EarlyStopping

In [155]:
train = pd.read_excel(path + "/Balanced/Balanced/Train.xlsx")
test= pd.read_excel(path + "/Balanced/Balanced/Test.xlsx")

In [156]:
classes = [
    "الطب العام",
    "امراض الاطفال",
    "امراض الجهاز التنفسي",
    "امراض الجهاز الهضمي",
    "امراض القلب و الشرايين"
]

In [157]:
df_train = train[train['category'].isin(classes)].copy()
df_test = test[test['category'].isin(classes)].copy()

print("Train:", df_train.shape)
print("Test :", df_test.shape)

print("\nTrain distribution:")
print(df_train['category'].value_counts())

print("\nTest distribution:")
print(df_test['category'].value_counts())

Train: (34000, 5)
Test : (8500, 5)

Train distribution:
category
الطب العام                6800
امراض الاطفال             6800
امراض الجهاز التنفسي      6800
امراض الجهاز الهضمي       6800
امراض القلب و الشرايين    6800
Name: count, dtype: int64

Test distribution:
category
الطب العام                1700
امراض الاطفال             1700
امراض الجهاز التنفسي      1700
امراض الجهاز الهضمي       1700
امراض القلب و الشرايين    1700
Name: count, dtype: int64


In [158]:
import re
import pandas as pd


def clean_arabic_text(text):
    text = str(text)

    # =========================
    # 1. Remove URLs
    # =========================
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    # =========================
    # 2. Remove HTML
    # =========================
    text = re.sub(r'<[^>]+>', ' ', text)

    # =========================
    # 3. Remove Tashkeel
    # =========================
    text = re.sub(
        r'[\u0617-\u061A\u064B-\u065F\u0670]',
        '',
        text
    )

    # =========================
    # 4. Normalize Arabic letters
    # =========================

    # Alef variations
    text = re.sub(r'[إأآٱ]', 'ا', text)

    # Alef Maqsura
    text = text.replace('ى', 'ي')

    # Remove Tatweel
    text = text.replace('ـ', '')

    # =========================
    # 5. Remove punctuation
    # Keep Arabic + English + numbers
    # =========================
    text = re.sub(r'[^\w\s]', ' ', text)

    # =========================
    # 6. Remove English letters
    # =========================
    text = re.sub(r'[A-Za-z]', ' ', text)

    # =========================
    # 7. Normalize numbers
    # =========================
    text = re.sub(r'\d+', ' ', text)

    # =========================
    # 8. Remove extra spaces
    # =========================
    text = re.sub(r'\s+', ' ', text).strip()

    return text


# ==========================================
# Apply preprocessing
# ==========================================

df_train['q_clean'] = df_train['q_body'].apply(clean_arabic_text)
df_train['a_clean'] = df_train['a_body'].apply(clean_arabic_text)

df_test['q_clean'] = df_test['q_body'].apply(clean_arabic_text)
df_test['a_clean'] = df_test['a_body'].apply(clean_arabic_text)


# ==========================================
# Combine Question + Answer
# ==========================================

df_train['text'] = (
    'سؤال ' +
    df_train['q_clean'] +
    ' جواب ' +
    df_train['a_clean']
)

df_test['text'] = (
    'سؤال ' +
    df_test['q_clean'] +
    ' جواب ' +
    df_test['a_clean']
)


# ==========================================
# Check results
# ==========================================

print(df_train[['q_body', 'q_clean', 'a_body', 'a_clean', 'category']]
      .head(5)
      .to_string(index=False))

                                                                                                                        q_body                                                                                                                     q_clean                                                                      a_body                                                                     a_clean   category
                   سلام عليكم اتعرض الي دوار عند وقت فطور في رمضان لمده ثلاث ايام والي الان وتكون عندي وزعلله في عين ولم اتحرك                 سلام عليكم اتعرض الي دوار عند وقت فطور في رمضان لمده ثلاث ايام والي الان وتكون عندي وزعلله في عين ولم اتحرك الدوار له نوعان محيطي ومركزي القصه غير كامله لذلك يجب التقيم عند طبيب عصبيه الدوار له نوعان محيطي ومركزي القصه غير كامله لذلك يجب التقيم عند طبيب عصبيه الطب العام
                هل ممكن ان تبقي النطف حيه علي ملابس بعد مرور 24 ساعه علما ان الملابس موضوعه في حقيبه سفر مغلقه ارجو الرد بسرعه                 هل ممكن ان تبقي النطف حيه علي

In [159]:
label_encoder = LabelEncoder()

df_train['label'] = label_encoder.fit_transform(
    df_train['category']
)

df_test['label'] = label_encoder.transform(
    df_test['category']
)

print("Classes:")
for i, cls in enumerate(label_encoder.classes_):
    print(i, "->", cls)

Classes:
0 -> الطب العام
1 -> امراض الاطفال
2 -> امراض الجهاز التنفسي
3 -> امراض الجهاز الهضمي
4 -> امراض القلب و الشرايين


In [160]:
MAX_WORDS = 30000

tokenizer = Tokenizer(
    num_words=MAX_WORDS,
    oov_token='<OOV>'
)

tokenizer.fit_on_texts(df_train['text'])

In [161]:
X_train = tokenizer.texts_to_sequences(
    df_train['text']
)

X_test = tokenizer.texts_to_sequences(
    df_test['text']
)

In [162]:

lengths = [len(x) for x in X_train]

print("Max length:", max(lengths))
print("Mean length:", np.mean(lengths))
print("95th percentile:", np.percentile(lengths, 95))

MAX_LEN = int(np.percentile(lengths, 95))

print("MAX_LEN:", MAX_LEN)

Max length: 168
Mean length: 36.59282352941177
95th percentile: 58.0
MAX_LEN: 58


In [163]:
MAX_LEN = int(np.percentile(lengths, 95))

print("MAX_LEN:", MAX_LEN)

MAX_LEN: 58


In [164]:
MAX_LEN = 56

In [165]:
X_train = pad_sequences(
    X_train,
    maxlen=MAX_LEN,
    padding='post',
    truncating='post'
)

X_test = pad_sequences(
    X_test,
    maxlen=MAX_LEN,
    padding='post',
    truncating='post'
)

print(X_train.shape)
print(X_test.shape)

(34000, 56)
(8500, 56)


In [166]:
y_train = df_train['label'].values
y_test = df_test['label'].values

print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

y_train: (34000,)
y_test : (8500,)


In [167]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.15,
    random_state=42,
    stratify=y_train
)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (28900, 56)
y_train: (28900,)
X_val: (5100, 56)
y_val: (5100,)
X_test: (8500, 56)
y_test: (8500,)


In [168]:
vocab_size = min(
    MAX_WORDS,
    len(tokenizer.word_index) + 1
)

print("Vocabulary size:", vocab_size)

Vocabulary size: 30000


#RNN

In [169]:
num_classes = len(label_encoder.classes_)

model = Sequential([

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    SimpleRNN(
        128,
        return_sequences=False
    ),


    Dense(
        64,
        activation='relu'
    ),

    Dense(
        num_classes,
        activation='softmax'
    )
])

In [170]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [171]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

In [172]:
import time

# =========================
# RNN Runtime
# =========================

rnn_start = time.time()

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=64,
    callbacks=[early_stop],
    verbose=1
)

rnn_runtime = time.time() - rnn_start

print(f"RNN Runtime: {rnn_runtime:.2f} seconds")

Epoch 1/15
452/452 ━━━━━━━━━━━━━━━━━━━━ 34s 68ms/step - accuracy: 0.5399 - loss: 1.0659 - val_accuracy: 0.6796 - val_loss: 0.8403
Epoch 2/15
452/452 ━━━━━━━━━━━━━━━━━━━━ 41s 68ms/step - accuracy: 0.7630 - loss: 0.6569 - val_accuracy: 0.7565 - val_loss: 0.7411
Epoch 3/15
452/452 ━━━━━━━━━━━━━━━━━━━━ 42s 70ms/step - accuracy: 0.8500 - loss: 0.4588 - val_accuracy: 0.7467 - val_loss: 0.7608
Epoch 4/15
452/452 ━━━━━━━━━━━━━━━━━━━━ 31s 69ms/step - accuracy: 0.8444 - loss: 0.4491 - val_accuracy: 0.7129 - val_loss: 0.9042
Epoch 5/15
452/452 ━━━━━━━━━━━━━━━━━━━━ 41s 68ms/step - accuracy: 0.8426 - loss: 0.4967 - val_accuracy: 0.7992 - val_loss: 0.6559
Epoch 6/15
452/452 ━━━━━━━━━━━━━━━━━━━━ 31s 68ms/step - accuracy: 0.8936 - loss: 0.3559 - val_accuracy: 0.8280 - val_loss: 0.6110
Epoch 7/15
452/452 ━━━━━━━━━━━━━━━━━━━━ 42s 69ms/step - accuracy: 0.9180 - loss: 0.2862 - val_accuracy: 0.8216 - val_loss: 0.7592
Epoch 8/15
452/452 ━━━━━━━━━━━━━━━━━━━━ 42s 70ms/step - accuracy: 0.9454 - loss: 0.1959 - 

In [173]:
test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=1
)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

266/266 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.8062 - loss: 0.6835
Test Loss: 0.6835
Test Accuracy: 0.8062


In [174]:
y_pred_prob = model.predict(X_test)

y_pred = np.argmax(
    y_pred_prob,
    axis=1
)

266/266 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step


In [175]:
print(
    classification_report(
        y_test,
        y_pred,
        target_names=label_encoder.classes_
    )
)

                        precision    recall  f1-score   support

            الطب العام       0.66      0.70      0.68      1700
         امراض الاطفال       0.88      0.86      0.87      1700
  امراض الجهاز التنفسي       0.82      0.82      0.82      1700
   امراض الجهاز الهضمي       0.82      0.83      0.82      1700
امراض القلب و الشرايين       0.88      0.82      0.85      1700

              accuracy                           0.81      8500
             macro avg       0.81      0.81      0.81      8500
          weighted avg       0.81      0.81      0.81      8500



#CNN

In [176]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    Conv1D,
    GlobalMaxPooling1D,
    Dense,
    Dropout
)

cnn_num_classes = len(label_encoder.classes_)

cnn_model = Sequential([

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    Conv1D(
        filters=128,
        kernel_size=3,
        activation='relu'
    ),

    GlobalMaxPooling1D(),

    Dense(
        128,
        activation='relu'
    ),

    Dropout(0.4),

    Dense(
        cnn_num_classes,
        activation='softmax'
    )
])

cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_model.summary()

Model: "sequential_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_12 (Embedding)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_2          │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_24 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_25 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [177]:
cnn_early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)
# =========================
# CNN Runtime
# =========================

cnn_start = time.time()

cnn_history = cnn_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=64,
    callbacks=[cnn_early_stop],
    verbose=1
)

cnn_runtime = time.time() - cnn_start



Epoch 1/15
452/452 ━━━━━━━━━━━━━━━━━━━━ 31s 63ms/step - accuracy: 0.7629 - loss: 0.6766 - val_accuracy: 0.8741 - val_loss: 0.3736
Epoch 2/15
452/452 ━━━━━━━━━━━━━━━━━━━━ 41s 64ms/step - accuracy: 0.9270 - loss: 0.2354 - val_accuracy: 0.8935 - val_loss: 0.3398
Epoch 3/15
452/452 ━━━━━━━━━━━━━━━━━━━━ 29s 64ms/step - accuracy: 0.9776 - loss: 0.0769 - val_accuracy: 0.8973 - val_loss: 0.3554
Epoch 4/15
452/452 ━━━━━━━━━━━━━━━━━━━━ 41s 64ms/step - accuracy: 0.9961 - loss: 0.0182 - val_accuracy: 0.8941 - val_loss: 0.4277
Epoch 5/15
452/452 ━━━━━━━━━━━━━━━━━━━━ 42s 66ms/step - accuracy: 0.9993 - loss: 0.0056 - val_accuracy: 0.8941 - val_loss: 0.4765


In [178]:
cnn_test_loss, cnn_test_accuracy = cnn_model.evaluate(
    X_test,
    y_test,
    verbose=1
)

print(f"CNN Test Loss: {cnn_test_loss:.4f}")
print(f"CNN Test Accuracy: {cnn_test_accuracy:.4f}")

266/266 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8749 - loss: 0.4050
CNN Test Loss: 0.4050
CNN Test Accuracy: 0.8749


In [179]:
cnn_y_pred_prob = cnn_model.predict(X_test)

cnn_y_pred = np.argmax(
    cnn_y_pred_prob,
    axis=1
)

266/266 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step


#BIRNN

In [180]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    SimpleRNN,
    Bidirectional,
    Dense,
    Dropout
)

birnn_model = Sequential([

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    Bidirectional(
        SimpleRNN(
            128,
            return_sequences=False
        )
    ),

    Dropout(0.3),

    Dense(
        64,
        activation='relu'
    ),

    Dropout(0.3),

    Dense(
        len(label_encoder.classes_),
        activation='softmax'
    )
])

birnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

birnn_model.summary()

Model: "sequential_13"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_13 (Embedding)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_17 (Dropout)            │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_26 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_18 (Dropout)            │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_27 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [181]:
birnn_early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

In [182]:
import time

birnn_start = time.time()

birnn_history = birnn_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=64,
    callbacks=[birnn_early_stop],
    verbose=1
)

birnn_runtime = time.time() - birnn_start

print(f"BiRNN Runtime: {birnn_runtime:.2f} seconds")
print(f"BiRNN Runtime: {birnn_runtime/60:.2f} minutes")

Epoch 1/15
452/452 ━━━━━━━━━━━━━━━━━━━━ 48s 99ms/step - accuracy: 0.5173 - loss: 1.1167 - val_accuracy: 0.7769 - val_loss: 0.6387
Epoch 2/15
452/452 ━━━━━━━━━━━━━━━━━━━━ 42s 93ms/step - accuracy: 0.8532 - loss: 0.4546 - val_accuracy: 0.8345 - val_loss: 0.4993
Epoch 3/15
452/452 ━━━━━━━━━━━━━━━━━━━━ 83s 96ms/step - accuracy: 0.9318 - loss: 0.2240 - val_accuracy: 0.8392 - val_loss: 0.5176
Epoch 4/15
452/452 ━━━━━━━━━━━━━━━━━━━━ 80s 92ms/step - accuracy: 0.9657 - loss: 0.1139 - val_accuracy: 0.8408 - val_loss: 0.6560
Epoch 5/15
452/452 ━━━━━━━━━━━━━━━━━━━━ 42s 92ms/step - accuracy: 0.9747 - loss: 0.0825 - val_accuracy: 0.8173 - val_loss: 0.7157
BiRNN Runtime: 295.54 seconds
BiRNN Runtime: 4.93 minutes


In [183]:
birnn_test_loss, birnn_test_accuracy = birnn_model.evaluate(
    X_test,
    y_test,
    verbose=1
)

print(f"BiRNN Test Loss: {birnn_test_loss:.4f}")
print(f"BiRNN Test Accuracy: {birnn_test_accuracy:.4f}")

266/266 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.8066 - loss: 0.5954
BiRNN Test Loss: 0.5954
BiRNN Test Accuracy: 0.8066


In [184]:
birnn_y_pred_prob = birnn_model.predict(X_test)

birnn_y_pred = np.argmax(
    birnn_y_pred_prob,
    axis=1
)

266/266 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step


In [185]:
results = pd.DataFrame({
    'Model': ['RNN', 'CNN', 'BiRNN'],

    'Train Accuracy': [
        max(history.history['accuracy']),
        max(cnn_history.history['accuracy']),
        max(birnn_history.history['accuracy'])
    ],

    'Validation Accuracy': [
        max(history.history['val_accuracy']),
        max(cnn_history.history['val_accuracy']),
        max(birnn_history.history['val_accuracy'])
    ],

    'Test Accuracy': [
        test_accuracy,
        cnn_test_accuracy,
        birnn_test_accuracy
    ],

    'Test Loss': [
        test_loss,
        cnn_test_loss,
        birnn_test_loss
    ],

    'Runtime (seconds)': [
        rnn_runtime,
        cnn_runtime,
        birnn_runtime
    ],

    'Epochs': [
        len(history.history['loss']),
        len(cnn_history.history['loss']),
        len(birnn_history.history['loss'])
    ]
})

results['Runtime (minutes)'] = results['Runtime (seconds)'] / 60

results

,Model,Train Accuracy,Validation Accuracy,Test Accuracy,Test Loss,Runtime (seconds),Epochs,Runtime (minutes)
0,RNN,0.956920,0.828039,0.806235,0.683453,333.183242,9,5.553054
1,CNN,0.999273,0.897255,0.874941,0.404974,183.693073,5,3.061551
2,BiRNN,0.974706,0.840784,0.806588,0.595377,295.541360,5,4.925689


#LSTM

In [186]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    LSTM,
    Dense,
    Dropout
)

lstm_model = Sequential([

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    LSTM(
        128,
        return_sequences=False
    ),

    Dropout(0.3),

    Dense(
        64,
        activation='relu'
    ),

    Dropout(0.3),

    Dense(
        len(label_encoder.classes_),
        activation='softmax'
    )
])

lstm_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

lstm_model.summary()

Model: "sequential_14"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_14 (Embedding)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_19 (Dropout)            │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_28 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_20 (Dropout)            │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_29 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [187]:
from tensorflow.keras.callbacks import EarlyStopping

lstm_early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

In [188]:
import time

lstm_start = time.time()

lstm_history = lstm_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=64,
    callbacks=[lstm_early_stop],
    verbose=1
)

lstm_runtime = time.time() - lstm_start

print(f"LSTM Runtime: {lstm_runtime:.2f} seconds")
print(f"LSTM Runtime: {lstm_runtime/60:.2f} minutes")

Epoch 1/15
452/452 ━━━━━━━━━━━━━━━━━━━━ 76s 162ms/step - accuracy: 0.5372 - loss: 1.0977 - val_accuracy: 0.7869 - val_loss: 0.6462
Epoch 2/15
452/452 ━━━━━━━━━━━━━━━━━━━━ 71s 158ms/step - accuracy: 0.8454 - loss: 0.5392 - val_accuracy: 0.8633 - val_loss: 0.4584
Epoch 3/15
452/452 ━━━━━━━━━━━━━━━━━━━━ 73s 161ms/step - accuracy: 0.9169 - loss: 0.3227 - val_accuracy: 0.8761 - val_loss: 0.4500
Epoch 4/15
452/452 ━━━━━━━━━━━━━━━━━━━━ 72s 159ms/step - accuracy: 0.9452 - loss: 0.2247 - val_accuracy: 0.8618 - val_loss: 0.4916
Epoch 5/15
452/452 ━━━━━━━━━━━━━━━━━━━━ 83s 162ms/step - accuracy: 0.9608 - loss: 0.1693 - val_accuracy: 0.8692 - val_loss: 0.4902
Epoch 6/15
452/452 ━━━━━━━━━━━━━━━━━━━━ 81s 161ms/step - accuracy: 0.9688 - loss: 0.1375 - val_accuracy: 0.8741 - val_loss: 0.4935
LSTM Runtime: 456.40 seconds
LSTM Runtime: 7.61 minutes


In [189]:
lstm_test_loss, lstm_test_accuracy = lstm_model.evaluate(
    X_test,
    y_test,
    verbose=1
)

print(f"LSTM Test Loss: {lstm_test_loss:.4f}")
print(f"LSTM Test Accuracy: {lstm_test_accuracy:.4f}")

266/266 ━━━━━━━━━━━━━━━━━━━━ 8s 30ms/step - accuracy: 0.8594 - loss: 0.5024
LSTM Test Loss: 0.5024
LSTM Test Accuracy: 0.8594


In [190]:
lstm_y_pred_prob = lstm_model.predict(X_test)

lstm_y_pred = np.argmax(
    lstm_y_pred_prob,
    axis=1
)

266/266 ━━━━━━━━━━━━━━━━━━━━ 8s 27ms/step


#GRU

In [191]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    GRU,
    Dense,
    Dropout
)

gru_model = Sequential([

    Embedding(
        input_dim=vocab_size,
        output_dim=128
    ),

    GRU(
        128,
        return_sequences=False
    ),

    Dropout(0.3),

    Dense(
        64,
        activation='relu'
    ),

    Dropout(0.3),

    Dense(
        len(label_encoder.classes_),
        activation='softmax'
    )
])

gru_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

gru_model.summary()

Model: "sequential_15"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_15 (Embedding)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_21 (Dropout)            │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_30 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_22 (Dropout)            │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_31 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [192]:
gru_early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

In [193]:
import time

gru_start = time.time()

gru_history = gru_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=15,
    batch_size=64,
    callbacks=[gru_early_stop],
    verbose=1
)

gru_runtime = time.time() - gru_start

print(f"GRU Runtime: {gru_runtime:.2f} seconds")
print(f"GRU Runtime: {gru_runtime/60:.2f} minutes")

Epoch 1/15
452/452 ━━━━━━━━━━━━━━━━━━━━ 76s 162ms/step - accuracy: 0.4542 - loss: 1.2215 - val_accuracy: 0.7625 - val_loss: 0.6925
Epoch 2/15
452/452 ━━━━━━━━━━━━━━━━━━━━ 71s 158ms/step - accuracy: 0.8492 - loss: 0.4743 - val_accuracy: 0.8639 - val_loss: 0.4425
Epoch 3/15
452/452 ━━━━━━━━━━━━━━━━━━━━ 73s 161ms/step - accuracy: 0.9239 - loss: 0.2586 - val_accuracy: 0.8680 - val_loss: 0.4706
Epoch 4/15
452/452 ━━━━━━━━━━━━━━━━━━━━ 72s 159ms/step - accuracy: 0.9562 - loss: 0.1567 - val_accuracy: 0.8700 - val_loss: 0.4961
Epoch 5/15
452/452 ━━━━━━━━━━━━━━━━━━━━ 73s 161ms/step - accuracy: 0.9719 - loss: 0.1034 - val_accuracy: 0.8631 - val_loss: 0.5519
GRU Runtime: 365.74 seconds
GRU Runtime: 6.10 minutes


In [194]:
gru_test_loss, gru_test_accuracy = gru_model.evaluate(
    X_test,
    y_test,
    verbose=1
)

print(f"GRU Test Loss: {gru_test_loss:.4f}")
print(f"GRU Test Accuracy: {gru_test_accuracy:.4f}")

266/266 ━━━━━━━━━━━━━━━━━━━━ 4s 16ms/step - accuracy: 0.8447 - loss: 0.4834
GRU Test Loss: 0.4834
GRU Test Accuracy: 0.8447


In [195]:
gru_y_pred_prob = gru_model.predict(X_test)

gru_y_pred = np.argmax(
    gru_y_pred_prob,
    axis=1
)

print(
    classification_report(
        y_test,
        gru_y_pred,
        target_names=label_encoder.classes_
    )
)

266/266 ━━━━━━━━━━━━━━━━━━━━ 6s 20ms/step
                        precision    recall  f1-score   support

            الطب العام       0.75      0.75      0.75      1700
         امراض الاطفال       0.93      0.90      0.91      1700
  امراض الجهاز التنفسي       0.82      0.91      0.86      1700
   امراض الجهاز الهضمي       0.85      0.83      0.84      1700
امراض القلب و الشرايين       0.89      0.82      0.85      1700

              accuracy                           0.84      8500
             macro avg       0.85      0.84      0.84      8500
          weighted avg       0.85      0.84      0.84      8500



In [196]:
results = pd.DataFrame({
    'Model': ['RNN', 'CNN', 'BiRNN', 'LSTM', 'GRU'],

    'Train Accuracy': [
        max(history.history['accuracy']),
        max(cnn_history.history['accuracy']),
        max(birnn_history.history['accuracy']),
        max(lstm_history.history['accuracy']),
        max(gru_history.history['accuracy'])
    ],

    'Validation Accuracy': [
        max(history.history['val_accuracy']),
        max(cnn_history.history['val_accuracy']),
        max(birnn_history.history['val_accuracy']),
        max(lstm_history.history['val_accuracy']),
        max(gru_history.history['val_accuracy'])
    ],

    'Test Accuracy': [
        test_accuracy,
        cnn_test_accuracy,
        birnn_test_accuracy,
        lstm_test_accuracy,
        gru_test_accuracy
    ],

    'Test Loss': [
        test_loss,
        cnn_test_loss,
        birnn_test_loss,
        lstm_test_loss,
        gru_test_loss
    ],

    'Runtime (seconds)': [
        rnn_runtime,
        cnn_runtime,
        birnn_runtime,
        lstm_runtime,
        gru_runtime
    ],

    'Epochs': [
        len(history.history['loss']),
        len(cnn_history.history['loss']),
        len(birnn_history.history['loss']),
        len(lstm_history.history['loss']),
        len(gru_history.history['loss'])
    ]
})

results['Runtime (minutes)'] = results['Runtime (seconds)'] / 60

results

,Model,Train Accuracy,Validation Accuracy,Test Accuracy,Test Loss,Runtime (seconds),Epochs,Runtime (minutes)
0,RNN,0.956920,0.828039,0.806235,0.683453,333.183242,9,5.553054
1,CNN,0.999273,0.897255,0.874941,0.404974,183.693073,5,3.061551
2,BiRNN,0.974706,0.840784,0.806588,0.595377,295.541360,5,4.925689
3,LSTM,0.968754,0.876078,0.859412,0.502384,456.403511,6,7.606725
4,GRU,0.971938,0.870000,0.844706,0.483444,365.736433,5,6.095607
